In [1]:
# -*- coding: utf-8 -*-
import os, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTENC
warnings.filterwarnings("ignore")

OUT = r"C:\Users\DELL\Desktop\代码结果"
os.makedirs(OUT, exist_ok=True)  # 若文件夹不存在则自动创建

X_train = pd.read_csv(os.path.join(OUT, "X_train.csv"))
X_test  = pd.read_csv(os.path.join(OUT, "X_test.csv"))
y_train = pd.read_csv(os.path.join(OUT, "y_train.csv"))["自杀意图"].astype(int).values
y_test  = pd.read_csv(os.path.join(OUT, "y_test.csv"))["自杀意图"].astype(int).values

prefix = ("性别_", "民族_", "生源地_", "是否独生_")
cat_idx = [i for i, c in enumerate(X_train.columns) if c.startswith(prefix)]
smote = SMOTENC(categorical_features=cat_idx, random_state=42, sampling_strategy=1.0)
X_bal, y_bal = smote.fit_resample(X_train, y_train)
print(f"✅ 数据就绪：训练集(增强后){X_bal.shape}，阳性占比{y_bal.mean():.2%}")

def full_metrics(y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "准确率": accuracy_score(y_true, y_pred),
        "精确率": precision_score(y_true, y_pred, zero_division=0),
        "召回率": recall_score(y_true, y_pred, zero_division=0),
        "特异度": tn / (tn + fp),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "NPV": tn / (tn + fn),
        "AUC-ROC": roc_auc_score(y_true, y_proba),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
    }

def tune_and_save(name, model, grid):
    print(f"\n⏳ 开始调参：{name}（共 {np.prod([len(v) for v in grid.values()])} 组参数）")
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    gs = GridSearchCV(model, grid, scoring="recall", cv=cv, n_jobs=-1, verbose=1)
    gs.fit(X_bal, y_bal)
    best = gs.best_estimator_
    y_proba = best.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)
    m = full_metrics(y_test, y_pred, y_proba)
    res = {"模型": name,
           "CV平均召回率": round(gs.cv_results_["mean_test_score"][gs.best_index_], 4),
           "最优参数": str(gs.best_params_)}
    for k, v in m.items():
        res[k] = v if k in ("TN","FP","FN","TP") else round(v, 4)
    save_path = os.path.join(OUT, f"调参结果_{name}.csv")
    pd.DataFrame([res]).to_csv(save_path, index=False, encoding="utf-8-sig")
    print(f"✅ {name} 完成，已保存 → {save_path}")
    print(f"   召回率={res['召回率']}  特异度={res['特异度']}  AUC={res['AUC-ROC']}")
    return best

print("准备完成，可逐个运行下面的 Cell。")

✅ 数据就绪：训练集(增强后)(3556, 28)，阳性占比50.00%
准备完成，可逐个运行下面的 Cell。


In [2]:
from sklearn.tree import DecisionTreeClassifier

dt = Pipeline([("model", DecisionTreeClassifier(random_state=42, splitter="best"))])
grid_dt = {
    "model__criterion": ["gini", "entropy"],
    "model__max_depth": [3, 4, 5],
    "model__class_weight": [{0:1,1:2}, {0:1,1:3}],
    "model__min_samples_leaf": [1, 2, 3],
}
tune_and_save("决策树", dt, grid_dt)


⏳ 开始调参：决策树（共 36 组参数）
Fitting 5 folds for each of 36 candidates, totalling 180 fits
✅ 决策树 完成，已保存 → C:\Users\DELL\Desktop\代码结果\调参结果_决策树.csv
   召回率=0.8987  特异度=0.557  AUC=0.7905


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](28,)","['学校适应困难','就业压力','学业压力',...,'生源地_大城市','生源地_小城镇','是否独生_是']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,28
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",3
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: dict, list of dict or ""balanced"", default=NoneWeights associated with classes in the form ``{class_label: weight}``.If None, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead 

In [3]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline([("model", RandomForestClassifier(random_state=42, n_jobs=-1))])
grid_rf = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [3, 4],
    "model__max_features": ["sqrt", "log2"],
    "model__min_samples_leaf": [1, 2, 3],
    "model__class_weight": ["balanced", {0:1,1:3}],
}
tune_and_save("随机森林", rf, grid_rf)


⏳ 开始调参：随机森林（共 72 组参数）
Fitting 5 folds for each of 72 candidates, totalling 360 fits
✅ 随机森林 完成，已保存 → C:\Users\DELL\Desktop\代码结果\调参结果_随机森林.csv
   召回率=0.9177  特异度=0.5203  AUC=0.8316


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](28,)","['学校适应困难','就业压力','学业压力',...,'生源地_大城市','生源地_小城镇','是否独生_是']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,28
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",3
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'log2'


In [4]:
from sklearn.ensemble import ExtraTreesClassifier

et = Pipeline([("model", ExtraTreesClassifier(random_state=42, n_jobs=-1))])
grid_et = {
    "model__n_estimators": [200, 300, 400],
    "model__max_depth": [3, 4],
    "model__max_features": ["sqrt", "log2"],
    "model__min_samples_leaf": [1, 2],
    "model__class_weight": ["balanced", {0:1,1:2}],
}
tune_and_save("ExtraTrees", et, grid_et)


⏳ 开始调参：ExtraTrees（共 48 组参数）
Fitting 5 folds for each of 48 candidates, totalling 240 fits
✅ ExtraTrees 完成，已保存 → C:\Users\DELL\Desktop\代码结果\调参结果_ExtraTrees.csv
   召回率=0.962  特异度=0.405  AUC=0.8354


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](28,)","['学校适应困难','就业压力','学业压力',...,'生源地_大城市','生源地_小城镇','是否独生_是']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,28
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",3
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'log2'


In [5]:
import xgboost as xgb

xg = Pipeline([("model", xgb.XGBClassifier(random_state=42, n_jobs=-1))])
grid_xgb = {
    "model__n_estimators": [100, 200, 300],
    "model__learning_rate": [0.03, 0.05],
    "model__max_depth": [3, 5],
    "model__min_child_weight": [1, 3],
    "model__scale_pos_weight": [2, 3],
    "model__subsample": [0.8],
    "model__colsample_bytree": [0.8],
}
tune_and_save("XGBoost", xg, grid_xgb)


⏳ 开始调参：XGBoost（共 48 组参数）
Fitting 5 folds for each of 48 candidates, totalling 240 fits
✅ XGBoost 完成，已保存 → C:\Users\DELL\Desktop\代码结果\调参结果_XGBoost.csv
   召回率=0.8354  特异度=0.6815  AUC=0.8518


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[<U7](28,)","['学校适应困难','就业压力','学业压力',...,'生源地_大城市','生源地_小城镇','是否独生_是']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,28
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None


In [6]:
import lightgbm as lgb

lg = Pipeline([("model", lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1))])
grid_lgb = {
    "model__n_estimators": [200, 300],
    "model__learning_rate": [0.03, 0.05],
    "model__max_depth": [3, 5],
    "model__num_leaves": [15, 20],
    "model__min_child_samples": [15, 20],
    "model__scale_pos_weight": [3, 4],
    "model__subsample": [0.8],
    "model__colsample_bytree": [0.8],
    "model__reg_alpha": [0.05],
    "model__reg_lambda": [1.5],
}
tune_and_save("LightGBM", lg, grid_lgb)


⏳ 开始调参：LightGBM（共 64 组参数）
Fitting 5 folds for each of 64 candidates, totalling 320 fits
✅ LightGBM 完成，已保存 → C:\Users\DELL\Desktop\代码结果\调参结果_LightGBM.csv
   召回率=0.8101  特异度=0.7012  AUC=0.8446


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[<U7](28,)","['学校适应困难','就业压力','学业压力',...,'生源地_大城市','生源地_小城镇','是否独生_是']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,28
,num_leaves,15
,max_depth,3
,learning_rate,0.03


In [7]:
print("\n📁 代码结果文件夹里的文件：")
for f in sorted(os.listdir(OUT)):
    print("  -", f)


📁 代码结果文件夹里的文件：
  - X_features.csv
  - X_test.csv
  - X_train.csv
  - cleaned_data.csv
  - merged_raw_preview.csv
  - split_info.csv
  - y_labels.csv
  - y_test.csv
  - y_train.csv
  - 调参结果_ExtraTrees.csv
  - 调参结果_LightGBM.csv
  - 调参结果_XGBoost.csv
  - 调参结果_决策树.csv
  - 调参结果_逻辑回归.csv
  - 调参结果_随机森林.csv
  - 逻辑回归_最终调参结果.csv
  - 逻辑回归_采样比例权衡.csv


In [ ]:
# -*- coding: utf-8 -*-
# 逻辑回归调参：纯recall评分
import os, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTENC
warnings.filterwarnings("ignore")

OUT = r"C:\Users\DELL\Desktop\代码结果"

# 加载数据
X_train = pd.read_csv(os.path.join(OUT, "X_train.csv"))
X_test  = pd.read_csv(os.path.join(OUT, "X_test.csv"))
y_train = pd.read_csv(os.path.join(OUT, "y_train.csv"))["自杀意图"].astype(int).values
y_test  = pd.read_csv(os.path.join(OUT, "y_test.csv"))["自杀意图"].astype(int).values

# SMOTE-ENC（采样比例固定1.0，与论文方法学一致）
prefix = ("性别_", "民族_", "生源地_", "是否独生_")
cat_idx = [i for i, c in enumerate(X_train.columns) if c.startswith(prefix)]
smote = SMOTENC(categorical_features=cat_idx, random_state=42, sampling_strategy=1.0)
X_bal, y_bal = smote.fit_resample(X_train, y_train)
print(f"增强后训练集 {X_bal.shape}，阳性占比 {y_bal.mean():.2%}")

# 模型
lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(penalty="elasticnet", solver="saga",
                                 max_iter=3000, random_state=42)),
])

# 最终参数网格：class_weight 上限 1:6
grid = {
    "model__C": [0.003, 0.004, 0.005, 0.006, 0.008],
    "model__l1_ratio": [0.28, 0.30, 0.32, 0.35, 0.38],
    "model__class_weight": [{0:1,1:5}, {0:1,1:6}],
}

# 纯召回率评分
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
gs = GridSearchCV(lr, grid, scoring="recall", cv=cv, n_jobs=-1, verbose=0)
gs.fit(X_bal, y_bal)

best = gs.best_estimator_
y_proba = best.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

# 全指标
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
metrics = {
    "CV平均召回率": round(gs.cv_results_["mean_test_score"][gs.best_index_], 4),
    "准确率": round(accuracy_score(y_test, y_pred), 4),
    "精确率": round(precision_score(y_test, y_pred, zero_division=0), 4),
    "召回率": round(recall_score(y_test, y_pred, zero_division=0), 4),
    "特异度": round(tn / (tn + fp), 4),
    "F1": round(f1_score(y_test, y_pred, zero_division=0), 4),
    "NPV": round(tn / (tn + fn), 4),
    "AUC-ROC": round(roc_auc_score(y_test, y_proba), 4),
    "TN": tn, "FP": fp, "FN": fn, "TP": tp,
}

print("\n===== 逻辑回归调参结果 =====")
print("最优参数:", gs.best_params_)
for k, v in metrics.items():
    print(f"{k}: {v}")

# 保存
res = {"最优参数": str(gs.best_params_), **metrics}
pd.DataFrame([res]).to_csv(
    os.path.join(OUT, "逻辑回归_最终调参结果.csv"),
    index=False, encoding="utf-8-sig"
)